In [2]:
import pandas as pd
import numpy as np 
import requests
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text 
from pathlib import Path

In [2]:
url = "https://www.basketball-reference.com/leagues/NBA_2026_games-october.html"


In [6]:
schedule = pd.read_html(url, attrs={"id" : "schedule"})[0]

In [12]:
schedule = schedule.rename(columns = {
    "Start (ET)" : "Start_Time",
    "Visitor/Neutral" : "Visitor",
    "PTS" : "Visitor_PTS",
    "Home/Neutral" : "Home",
    "PTS.1" : "Home_PTS",
    "Unnamed: 7" : "Overtime",
    "Attend." : "Attendance",
    "Notes" : "Game_Notes"
})

In [16]:
schedule = schedule.drop(columns=["LOG", "Unnamed: 6"], errors="ignore")
schedule["Game_Notes"] = schedule["Game_Notes"].fillna("None")
schedule["Overtime"] = schedule["Overtime"].fillna("None")

In [19]:
schedule

,Date,Start_Time,Visitor,Visitor_PTS,Home,Home_PTS,Overtime,Attendance,Arena,Game_Notes
0,"Tue, Oct 21, 2025",7:30p,Houston Rockets,124,Oklahoma City Thunder,125,2OT,18203,Paycom Center,None
1,"Tue, Oct 21, 2025",10:00p,Golden State Warriors,119,Los Angeles Lakers,109,None,18997,Crypto.com Arena,None
2,"Wed, Oct 22, 2025",7:00p,Brooklyn Nets,117,Charlotte Hornets,136,None,19516,Spectrum Center,None
3,"Wed, Oct 22, 2025",7:00p,Cleveland Cavaliers,111,New York Knicks,119,None,19812,Madison Square Garden (IV),None
4,"Wed, Oct 22, 2025",7:00p,Miami Heat,121,Orlando Magic,125,None,19186,Kia Center,None
...,...,...,...,...,...,...,...,...,...,...
75,"Fri, Oct 31, 2025",8:00p,New York Knicks,125,Chicago Bulls,135,None,18330,United Center,NBA Cup
76,"Fri, Oct 31, 2025",9:30p,Los Angeles Lakers,117,Memphis Grizzlies,112,None,16122,FedExForum,NBA Cup
77,"Fri, Oct 31, 2025",10:00p,Utah Jazz,96,Phoenix Suns,118,None,17071,Mortgage Matchup Center,NBA Cup
78,"Fri, Oct 31, 2025",10:00p,Denver Nuggets,107,Portland Trail Blazers,109,None,16382,Moda Center,NBA Cup


In [22]:
schedule["Date"] = pd.to_datetime(schedule["Date"])

s = schedule["Start_Time"].str.strip().str.lower()
s = s.str.replace(r"(\d)(a)$", r"\1AM", regex=True)
s = s.str.replace(r"(\d)(p)$", r"\1PM", regex=True)

schedule["Start_Time"] = pd.to_datetime(s, format="%I:%M%p", errors="coerce")

In [24]:
schedule["Start_Time"] = schedule["Start_Time"].dt.time

,Date,Start_Time,Visitor,Visitor_PTS,Home,Home_PTS,Overtime,Attendance,Arena,Game_Notes
0,2025-10-21,19:30:00,Houston Rockets,124,Oklahoma City Thunder,125,2OT,18203,Paycom Center,None
1,2025-10-21,22:00:00,Golden State Warriors,119,Los Angeles Lakers,109,None,18997,Crypto.com Arena,None
2,2025-10-22,19:00:00,Brooklyn Nets,117,Charlotte Hornets,136,None,19516,Spectrum Center,None
3,2025-10-22,19:00:00,Cleveland Cavaliers,111,New York Knicks,119,None,19812,Madison Square Garden (IV),None
4,2025-10-22,19:00:00,Miami Heat,121,Orlando Magic,125,None,19186,Kia Center,None
...,...,...,...,...,...,...,...,...,...,...
75,2025-10-31,20:00:00,New York Knicks,125,Chicago Bulls,135,None,18330,United Center,NBA Cup
76,2025-10-31,21:30:00,Los Angeles Lakers,117,Memphis Grizzlies,112,None,16122,FedExForum,NBA Cup
77,2025-10-31,22:00:00,Utah Jazz,96,Phoenix Suns,118,None,17071,Mortgage Matchup Center,NBA Cup
78,2025-10-31,22:00:00,Denver Nuggets,107,Portland Trail Blazers,109,None,16382,Moda Center,NBA Cup


In [ ]:
db_path = Path("~/Personal Project/data/nba.db").expanduser()
engine = create_engine(f"sqlite:///{db_path}")

schedule.to_sql(
    "game_schedule",
    engine,
    if_exists="append",
    index=False
)

In [32]:
url = "https://www.basketball-reference.com/leagues/NBA_1951_games-november.html"

df = pd.read_html(url, attrs={"id" : "schedule"})[0]
df["Start (ET)"].isna().sum() == len(df)

np.True_

In [6]:
db_path = Path("~/Personal Project/data/nba.db").expanduser()
engine = create_engine(f"sqlite:///{db_path}")

df = pd.read_sql(text("SELECT * FROM game_schedule_history"), engine)

In [7]:
df

,Date,Visitor,Visitor_PTS,Home,Home_PTS,Overtime,Attendance,Arena,Game_Notes
